In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from rich import print as rich_print

from helpers import classify_cage_search_result
from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.basis.sectors import sector_mask_from_build_result
from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    classify_cage_state,
    build_type1_cage_lindblad_construction,
)
from qlinks.models import SquareQuantumDiskModel
from qlinks.visualizer import (
    HamiltonianGraphStyle,
    HamiltonianGraphVisualizer,
    QuantumDiskConfigurationVisualizer,
    QuantumDiskBasisGridVisualizer,
    QuantumDiskVisualStyle,
    plot_quantum_disk_basis_grid,
)

## Model definition

In [ ]:
model = SquareQuantumDiskModel(
    lx=5,
    ly=5,
    boundary_condition="open",
    disk_number=4,
    coup_kin=-1.0,
    coup_pot=1.0,  # Count the number of movable disks
    chemical_potential=0.0,  # Count the total number of disks
    hard_core_nearest_neighbor=True,  # Hard-core nearest-neighbor blockade.
    hop_families=("x_plus_y",),  # Diagonal hopping family, "x_plus_y" means hops along (+1, -1), preserving x + y.
    # x_plus_y_sums=(0, 0, 1, 0, 1, 0, 0),  # Number of disks along each diagonal/off-diagonal
)

print(model.allowed_sector_labels())
print(model.nonempty_sector_labels())

In [ ]:
build_result = model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
    on_missing="raise",
)

hamiltonian_matrix = build_result.hamiltonian
kinetic_matrix = build_result.kinetic
potential_matrix = build_result.potential
basis = build_result.basis
basis_configs = basis_configs_from_build_result(build_result)
sector_mask = sector_mask_from_build_result(build_result)

print("n_states =", basis.n_states)
print("H shape =", hamiltonian_matrix.shape)
print("H nnz =", hamiltonian_matrix.nnz)
print("K nnz =", kinetic_matrix.nnz)
print("V nnz =", potential_matrix.nnz)
print("K is bipartite =", nx.is_bipartite(nx.from_scipy_sparse_array(kinetic_matrix, edge_attribute="weight")))

## Search for caged states

In [ ]:
searcher = CageSearcher.from_model_build_result(
    build_result,
    config=CageSearchConfig(
        search_type="type1",
        # type1_kappas=(0,),
        # type2_kappas=(-2, 2),
        tolerance=1e-10,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=256,
        ipr_candidate_count=128,
        ipr_random_seed=1234,
        store_full_states=True,
    ),
)

search_result = searcher.run()
print("number of cages:", len(search_result))
print("signatures:", search_result.signatures)
print("counts by signature:", search_result.counts_by_signature)

In [ ]:
signature = (0, 4)
record_index = 21
record = search_result[signature, record_index]

state_vector = record.full_state
if state_vector is None:
    state_vector = search_result.full_state_matrix()[record_index]

print("selected signature:", record.signature)
print("support size:", record.support.size)
print("energy:", record.cage_state.energy)

## Classify the selected caged state

In [ ]:
report = classify_cage_state(
    record.cage_state,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis_configs,
    sector_mask=sector_mask,
    hilbert_size=search_result.hilbert_size,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
        sector_policy="infer_support_component",
    ),
)

# rich_print(report)
rich_print(report.to_text(verbose=True))

## Visualize the caged state on Hamiltonian graph

In [ ]:
graph_visualizer = HamiltonianGraphVisualizer.from_sparse_matrix(
    kinetic_matrix,
    include_self_loops=False,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=24,
        vertex_label_size=4,
    ),
)

graph_visualizer.plot(
    backend="igraph-mpl",
    color_by="state_amplitude_real",
    edge_color_by="weight_phase",
    state_vector=state_vector,
    layout="mds",
    title=f"Fock-space graph colored by cage-state signed amplitude",
)

In [ ]:
small_viz = HamiltonianGraphVisualizer.cage_subgraph_from_sparse_matrix(
    build_result.kinetic,
    state_vector,
    classification_report=report,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=48,
        vertex_label_size=6,
    ),
)

small_viz.plot(
    backend="igraph-mpl",
    color_by="state_amplitude_real",
    edge_color_by="weight_phase",
    layout="kk",
    title=f"The relevant subgraph colored by cage-state signed amplitude",
)

# Visualize the basis states

In [ ]:
disk_viz = QuantumDiskBasisGridVisualizer.from_model(
    model,
    site_label_style="cell",
    style=QuantumDiskVisualStyle(
        site_label_fontsize=5.0,
        site_marker_color="lightgray",
    )
)

In [ ]:
# Plot cage support
disk_viz.plot_cage_support(
    search_result,
    basis_configs=basis_configs,
    signature=signature,
    record_index=record_index,
    ncols=4,
)

In [ ]:
# Plot interference zeros
disk_viz.plot_interference_zeros(
    report,
    basis_configs=basis_configs,
    mechanism="all",
    ncols=4,
)

In [ ]:
# Plot all basis states
disk_viz.plot_basis(
    basis_configs,
    ncols=4,
)

## Lindblad dynamics

In [ ]:
problem = build_type1_cage_lindblad_construction(
    model=model,
    build_result=build_result,
    cage_state=state_vector,
    classification_report=report,
    z_value=signature[1],
    monitor_source="reduced_iz_operators",
    reduced_iz_monitor_content="offdiagonal_only",
    reduced_iz_monitor_decomposition="exact_support",
    jump_operator_design="kinetic_outside_monitor_inside",
    recycling_jump_source="local_rdm_block_reset",
    deduplicate_recycling_regions=True,
    kinetic_jump_grouping="support_greedy",
    max_kinetic_jump_support_size=8,
    check_liouvillian=True,
)

rich_print(problem.to_rich())

In [ ]:
df, classification_reports = classify_cage_search_result(
    search_result,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis_configs,
    sector_mask=None,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
        sector_policy="infer_support_component",
        collective_cancellation_mode="all_problematic_nullspace",
    ),
)

df